# Workshop 3 · 00 — Setup

Erzeugt die Tabelle `asset_meldungen` mit **bewusst uneinheitlichen und unvollständigen** Daten:
- Stammdaten teils leer / uneinheitlich (`baureihe`, `depot`), `komponente` fehlt
- `freitext_meldung` (Werkstatt-Freitext — die Struktur steckt nur im Text)
- `kunden_kommentar` (Material für Sentiment & Keywords)
- `foto_pfad` (Verweis auf Wagon-Fotos für den CV-Teaser)

**Voraussetzung:** Ein Lakehouse ist an dieses Notebook angehängt.

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType
import random
from datetime import date, timedelta

random.seed(7)

# Absichtlich uneinheitliche Stammdaten
baureihen = ["ICE 4", "ice4", "ICE-4", "ICE3", "ic 2", None, "Talent 2", "FLIRT 3", None, "Doppelstock"]
depots = ["HH-Eidelstedt", "Hamburg Eidelstedt", "hamburg-eidelstedt", "FFM Griesheim",
          "Frankfurt-Griesheim", "Muenchen Pasing", "muenchen pasing", "Leipzig Hbf", "Berlin Rummelsburg"]

# Im Freitext steckt die Wahrheit: Komponente, Ort, Schweregrad
freitexte = [
    "Tuer 2 klemmt beim Schliessen, leichte Roststelle am Tuerrahmen, nicht sicherheitskritisch",
    "Riss in der Seitenwand Wagen 3, dringend pruefen, sicherheitsrelevant",
    "Graffiti grossflaechig an der Aussenwand, rein kosmetisch",
    "Klimaanlage Abteil 5 ohne Funktion, mehrere Beschwerden",
    "Delle am Drehgestell nach Steinschlag, Strukturpruefung empfohlen",
    "WC im Wagen 1 defekt, Geruchsbelaestigung, mittlere Prioritaet",
    "Fenster laesst sich nicht verriegeln, Sicherheitsrisiko bei Fahrt",
    "Sitzpolster Reihe 12 eingerissen, kosmetisch",
    "Bremsbelag stark abgenutzt, sofort in die Werkstatt",
    "Beleuchtung Eingangsbereich flackert, geringe Prioritaet",
]
kommentare = [
    "Puenktlich und sauber, sehr zufrieden!",
    "WLAN ging die ganze Fahrt nicht, aergerlich.",
    "Klimaanlage viel zu kalt, unangenehm.",
    "Freundliches Personal, gerne wieder.",
    "Zug 20 Minuten zu spaet, Anschluss verpasst.",
    "Sitze bequem, aber sehr laut im Wagen.",
    "Toilette war defekt und verschmutzt.",
    "Top Reise, alles bestens.",
]

# Explizites Schema -> komponente/foto_pfad sind komplett bzw. teils None,
# sonst kann Spark den Typ nicht ableiten ([CANNOT_DETERMINE_TYPE]).
schema = StructType([
    StructField("meldung_id",       StringType(), False),
    StructField("wagon_id",         StringType(), False),
    StructField("baureihe",         StringType(), True),
    StructField("depot",            StringType(), True),
    StructField("komponente",       StringType(), True),
    StructField("freitext_meldung", StringType(), True),
    StructField("kunden_kommentar", StringType(), True),
    StructField("foto_pfad",        StringType(), True),
    StructField("eingang",          StringType(), True),
])

rows = []
for i in range(24):
    foto = ("Files/wagon_photos_w3/W3-%02d.png" % i) if i < 6 else None
    rows.append(Row(
        meldung_id="M-%03d" % i,
        wagon_id="W3-%02d" % i,
        baureihe=random.choice(baureihen),   # teils None / uneinheitlich
        depot=random.choice(depots),         # frei geschrieben
        komponente=None,                     # fehlt -> spaeter aus Freitext gezogen
        freitext_meldung=freitexte[i % len(freitexte)],
        kunden_kommentar=kommentare[i % len(kommentare)],
        foto_pfad=foto,
        eingang=(date(2026, 1, 1) + timedelta(days=random.randint(0, 160))).isoformat(),
    ))

sdf = spark.createDataFrame(rows, schema=schema)
sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("asset_meldungen")
print("Zeilen:", sdf.count())
display(sdf)


In [ ]:
import os, io, zipfile, random, urllib.request, urllib.error
from PIL import Image, ImageDraw

OUT = "/lakehouse/default/Files/wagon_photos_w3"
os.makedirs(OUT, exist_ok=True)
random.seed(7)

# --- 1) Bilder aus GitHub-Repo laden -----------------
GH_OWNER  = "henrik-ra"
GH_REPO   = "fabric-ai-workshop-images"   # oeffentliches Repo mit den Bildern
GH_BRANCH = "main"
GH_FOLDER = "wagon"                        # Unterordner im Repo (leer = ganzes Repo)

UA = {"User-Agent": "fabric-ai-workshop/1.0 (workshop demo)"}
ZIP_URL = "https://github.com/%s/%s/archive/refs/heads/%s.zip" % (GH_OWNER, GH_REPO, GH_BRANCH)
EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

geladen = set()
try:
    req = urllib.request.Request(ZIP_URL, headers=UA)
    with urllib.request.urlopen(req, timeout=60) as r:
        blob = r.read()
    zf = zipfile.ZipFile(io.BytesIO(blob))
    # Bild-Eintraege im gewuenschten Ordner einsammeln (Zip-Root = repo-branch/)
    marker = ("/%s/" % GH_FOLDER) if GH_FOLDER else "/"
    treffer = sorted(
        n for n in zf.namelist()
        if not n.endswith("/") and marker in n and n.lower().endswith(EXTS)
    )
    for i, name in enumerate(treffer[:6]):
        img = Image.open(io.BytesIO(zf.read(name))).convert("RGB")
        img.save(OUT + "/W3-%02d.png" % i)
        geladen.add(i)
        print("geladen:", name.split("/")[-1])
    if not treffer:
        print("Repo erreichbar, aber keine Bilder in '%s' gefunden -> synthetisch" % GH_FOLDER)
except Exception as e:
    print("Repo-Download fehlgeschlagen (%s) -> synthetisch" % type(e).__name__)

# --- 2) Fehlende Bilder synthetisch auffuellen (Fallback, klappt offline) --
# schaden = ["rost", "kratzer", "delle", "graffiti", "riss", "kein_schaden"]
# for i in range(6):
#     if i in geladen:
#         continue
#     img = Image.new("RGB", (640, 360), random.choice([(200, 30, 40), (230, 230, 230), (40, 60, 90)]))
#     d = ImageDraw.Draw(img)
#     for x in range(0, 640, 80):
#         d.line([x, 0, x, 360], fill=(0, 0, 0), width=1)
#     kind = schaden[i % len(schaden)]
#     if kind == "rost":
#         for _ in range(40):
#             x, y = random.randint(0, 640), random.randint(0, 360); r = random.randint(2, 9)
#             d.ellipse([x, y, x + r, y + r], fill=(120, 60, 20))
#     elif kind == "kratzer":
#         for _ in range(3):
#             x, y = random.randint(0, 640), random.randint(0, 360)
#             d.line([x, y, x + random.randint(-150, 150), y + random.randint(-20, 20)], fill=(20, 20, 20), width=2)
#     elif kind == "graffiti":
#         for _ in range(4):
#             pts = [(random.randint(0, 640), random.randint(0, 360))]
#             for _ in range(6):
#                 pts.append((pts[-1][0] + random.randint(-50, 50), pts[-1][1] + random.randint(-30, 30)))
#             d.line(pts, fill=(0, 200, 80), width=6)
#     elif kind == "riss":
#         pts = [(random.randint(0, 640), 0)]
#         for _ in range(8):
#             pts.append((pts[-1][0] + random.randint(-20, 20), pts[-1][1] + 45))
#         d.line(pts, fill=(10, 10, 10), width=2)
#     elif kind == "delle":
#         x, y = random.randint(80, 560), random.randint(80, 280); r = random.randint(25, 50)
#         d.ellipse([x - r, y - r, x + r, y + r], outline=(70, 70, 70), width=4)
#     d.rectangle([18, 318, 250, 346], fill=(255, 255, 255))
#     d.text((24, 326), "UIC 9480123456%02d" % i, fill=(0, 0, 0))
#     img.save(OUT + "/W3-%02d.png" % i)

print("Fotos bereit in", OUT, "| echt:", len(geladen), "synthetisch:", 6 - len(geladen))


> **Bilder aus einem GitHub-Repo:** Die Zelle oben lädt ein **öffentliches** GitHub-Repo als ZIP in einem einzigen Request (kein Login, kein Token, kein Rate-Limit) und entpackt die Bilder aus dem Ordner `GH_FOLDER`. Die ersten sechs (alphabetisch) werden zu `W3-00.png … W3-05.png`. Schlägt der Download fehl, greift der synthetische Fallback.
>
> **Lizenzgeprüfte Startsammlung** (im Repo unter `data/sample_photos/`): sechs Bilder von Wikimedia Commons, je eines pro Schadensklasse — `00_rost` (CC BY-SA 2.0), `01_kratzer` (CC0), `02_delle` (CC0), `03_graffiti` (CC BY-SA 4.0), `04_riss` (Public Domain), `05_kein_schaden` (CC BY-SA 2.0). Quellen/Attribution in `data/sample_photos/CREDITS.md`.
>
> **Vorbereitung:**
> 1. Inhalt von `data/sample_photos/` in ein öffentliches GitHub-Repo legen (Ordner = `GH_FOLDER`).
> 2. `GH_OWNER` / `GH_REPO` / `GH_BRANCH` / `GH_FOLDER` anpassen.
> 3. Beim nächsten Lauf zieht das Notebook die echten Bilder automatisch (Reihenfolge `00…05` entspricht rost/kratzer/delle/graffiti/riss/kein_schaden).
>
> **Production:** Größere Foto-/Dokumentbestände aus einem ADLS-Container per **OneLake Shortcut** einbinden (*Lakehouse → Files → New shortcut*) — kein Kopieren, ein Governance-Modell.